# Bilge Pump System — Native SysML v2 Analysis

This notebook uses the **SysML v2 Jupyter kernel** (`jupyter-sysml-kernel 0.58.0`).
Each code cell contains real SysML v2 source — the kernel parses and evaluates it natively.
No Python approximation. The kernel is the same evaluator used by the SysML v2 Pilot Implementation.

| Item | Value |
|---|---|
| Kernel | SysML v2 (`sysml`) |
| API backend | `http://sysml2.intercax.com:9000` (pre-configured in kernel.json) |
| Java required | 21+ (installed at `/usr/lib/jvm/java-21-openjdk-amd64`) |

## Workflow
- Cells **1–4** define the type vocabulary, architecture, requirements, and analysis
- Cell **5** runs `BilgePumpVerification` — the positive test
- Cell **6** runs the negative test (pump A failure)
- Use `%publish` at the end of any cell to push the model to the SST API

## Relationship to the other files
The SysML source in these cells mirrors `Library.sysml`, `Architecture.sysml`,
`Requirements.sysml`, and `Analysis.sysml`. Those `.sysml` files are the canonical
source committed to the API via `bash commit.sh`. This notebook is the native
execution and exploration environment.

> **First run:** the kernel loads its standard library (~5–10 s). Subsequent cells are faster.

In [2]:
// ============================================================
// Cell 1 — Library: part def, port def, attribute def
// Type vocabulary for the entire bilge pump system.
// ============================================================

package BilgePump_Library {
    private import ScalarValues::*;

    // --- Port definitions ---
    port def LevelSignalPort  { attribute waterLevel : Real; }
    port def ControlPort      { attribute controlValue : Real; }
    port def PowerPort        { attribute voltage    : Real; }
    port def FluidFlowPort    { attribute flowRate   : Real; }
    port def StatusPort       { attribute statusCode : Real; }
    port def AlarmPort        { attribute alarmCode  : Real; }
    port def NotifyPort       { attribute notifyText : Real; }

    // --- Part definitions ---
    part def BilgeWaterSensor {
        port levelOut   : LevelSignalPort;
        attribute waterLevel        : Real;
        attribute triggerThreshold  : Real;
    }

    part def PumpController {
        port levelIn        : ~LevelSignalPort;
        port pumpAControl   : ControlPort;
        port pumpBControl   : ControlPort;
        port statusOut      : StatusPort;
        port overrideIn     : ~ControlPort;
        port alarmOut       : AlarmPort;
        attribute triggerLevel_m    : Real;
        attribute responseTime_s    : Real;
    }

    part def PowerSupply {
        port powerOutA      : PowerPort;
        port powerOutB      : PowerPort;
        attribute nominalVoltage    : Real;
        attribute redundancyActive  : Boolean;
    }

    part def BilgePumpA {
        port controlIn      : ~ControlPort;
        port powerIn        : ~PowerPort;
        port flowOut        : FluidFlowPort;
        attribute flowRate   : Real;
        attribute efficiency : Real;
        attribute runHours   : Real;
    }

    part def BilgePumpB {
        port controlIn      : ~ControlPort;
        port powerIn        : ~PowerPort;
        port flowOut        : FluidFlowPort;
        attribute flowRate   : Real;
        attribute efficiency : Real;
        attribute runHours   : Real;
        attribute isRedundant : Boolean;
    }

    part def DischargeLine {
        port flowInA        : ~FluidFlowPort;
        port flowInB        : ~FluidFlowPort;
        attribute pipeLossFactor : Real;
    }

    part def AlarmSystem {
        port alarmIn        : ~AlarmPort;
        port notifyOut      : NotifyPort;
        attribute activationDelay_s : Real;
        attribute isActive          : Boolean;
    }

    part def OperatorInterface {
        port statusIn       : ~StatusPort;
        port overrideOut    : ControlPort;
        port notifyIn       : ~NotifyPort;
        attribute overrideActive : Boolean;
    }
}


Package BilgePump_Library (576e807a-f8f4-405d-8581-5adfb3d76e16)


In [3]:
// ============================================================
// Cell 2 — Architecture: BilgePumpSystem composition
// 8 part usages, 11 connect statements.
// ============================================================

package BilgePump_Architecture {
    private import BilgePump_Library::*;

    part def BilgePumpSystem {

        part sensor     : BilgeWaterSensor;
        part controller : PumpController;
        part power      : PowerSupply;
        part pumpA      : BilgePumpA;
        part pumpB      : BilgePumpB;
        part discharge  : DischargeLine;
        part alarm      : AlarmSystem;
        part ui         : OperatorInterface;

        // Signal flow connections
        connect sensor.levelOut         to controller.levelIn;
        connect controller.pumpAControl to pumpA.controlIn;
        connect controller.pumpBControl to pumpB.controlIn;
        connect power.powerOutA         to pumpA.powerIn;
        connect power.powerOutB         to pumpB.powerIn;
        connect pumpA.flowOut           to discharge.flowInA;
        connect pumpB.flowOut           to discharge.flowInB;
        connect controller.statusOut    to ui.statusIn;
        connect ui.overrideOut          to controller.overrideIn;
        connect controller.alarmOut     to alarm.alarmIn;
        connect alarm.notifyOut         to ui.notifyIn;
    }
}


Package BilgePump_Architecture (0d0cd593-0fbe-4e1f-b0ee-69fe636aaaff)


In [4]:
// ============================================================
// Cell 3 — Requirements
// 4 requirement def blocks with require constraint bodies.
// Regulatory basis: IMO MARPOL, DNV, IEC 60945, SOLAS.
// ============================================================

package BilgePump_Requirements {
    private import ScalarValues::*;
    private import BilgePump_Library::*;
    private import BilgePump_Architecture::*;

    // BPS-REQ-001: Water Level Threshold (IMO MARPOL 73/78 Annex I Reg. 22)
    requirement def WaterLevelRequirement {
        subject sys : BilgePumpSystem;
        doc /* Bilge water level shall not exceed 300 mm above bilge floor
               during normal automated operation. */
        require constraint { sys.sensor.waterLevel <= 0.3 }
    }

    // BPS-REQ-002: Pump Redundancy (DNV Rules Part 4 Ch. 6, SOLAS II-1 Reg. 35)
    requirement def PumpRedundancyRequirement {
        subject sys : BilgePumpSystem;
        doc /* Pump B shall be designated redundant with independent power feed. */
        require constraint { sys.pumpB.isRedundant == true }
    }

    // BPS-REQ-003: Alarm Response Time (IEC 60945:2002 Section 4.3)
    requirement def AlarmResponseRequirement {
        subject sys : BilgePumpSystem;
        doc /* Alarm shall activate within 2.0 seconds of controller trigger. */
        require constraint { sys.alarm.activationDelay_s <= 2.0 }
    }

    // BPS-REQ-004: Discharge Capacity (SOLAS II-1 Reg. 35)
    requirement def DischargeCapacityRequirement {
        subject sys : BilgePumpSystem;
        attribute designInflow : Real;
        doc /* Combined pump discharge shall meet or exceed design inflow rate. */
        require constraint {
            (sys.pumpA.flowRate + sys.pumpB.flowRate) >= designInflow
        }
    }
}


Package BilgePump_Requirements (caba2b99-6181-4472-b268-3f3f4604c405)


In [5]:
// ============================================================
// Cell 4 — Analysis: physics constraint + analysis def
//
// PumpFlowPhysics: Q_net = (Q_A + Q_B) × η × (1 − λ)
// BilgePumpVerification: the test runner
// ============================================================

package BilgePump_Analysis {
    private import ScalarValues::*;
    private import BilgePump_Library::*;
    private import BilgePump_Architecture::*;
    private import BilgePump_Requirements::*;

    constraint def PumpFlowPhysics {
        in attribute flowRateA      : Real;
        in attribute flowRateB      : Real;
        in attribute efficiency     : Real;
        in attribute pipeLossFactor : Real;
        in attribute designInflow   : Real;

        (flowRateA + flowRateB) * efficiency * (1.0 - pipeLossFactor) >= designInflow
    }

    analysis def BilgePumpVerification {
        subject sys : BilgePumpSystem;
        in attribute pumpAFlowRate   : Real;
        in attribute pumpBFlowRate   : Real;
        in attribute pumpEfficiency  : Real;
        in attribute pumpARunHours   : Real;
        in attribute pumpBRunHours   : Real;
        in attribute pipeLossFactor  : Real;
        in attribute designInflow    : Real;

        objective bilgePumpObjective {
            assume constraint {
                sys.sensor.waterLevel == 0.15 &
                sys.controller.triggerLevel_m == 0.25 &
                sys.controller.responseTime_s == 1.0 &
                sys.power.nominalVoltage == 440.0 &
                sys.power.redundancyActive == false &
                sys.pumpA.flowRate == pumpAFlowRate &
                sys.pumpA.efficiency == pumpEfficiency &
                sys.pumpA.runHours == pumpARunHours &
                sys.pumpB.flowRate == pumpBFlowRate &
                sys.pumpB.efficiency == pumpEfficiency &
                sys.pumpB.runHours == pumpBRunHours &
                sys.pumpB.isRedundant == true &
                sys.discharge.pipeLossFactor == pipeLossFactor &
                sys.alarm.activationDelay_s == 0.5 &
                sys.alarm.isActive == false &
                sys.ui.overrideActive == false &
                sys.sensor.waterLevel <= 0.3 &
                sys.alarm.activationDelay_s <= 2.0 &
                sys.pumpB.isRedundant == true
            }

            assert constraint physicsCheck {
                PumpFlowPhysics(
                    flowRateA = sys.pumpA.flowRate,
                    flowRateB = sys.pumpB.flowRate,
                    efficiency = sys.pumpA.efficiency,
                    pipeLossFactor = sys.discharge.pipeLossFactor,
                    designInflow = designInflow
                )
            }
        }
    }
}


Package BilgePump_Analysis (7f0127f1-fd3b-4cb1-a4a1-7fed83470482)


In [6]:
// ============================================================
// Cell 5 — POSITIVE TEST: Nominal values
//
// All parameters at design-nominal. All 4 requirements expected
// to be SATISFIED.
//
// Q_net = (0.025 + 0.025) × 0.82 × (1 − 0.05) = 0.0389 m³/s
// ============================================================

analysis positiveTest : BilgePump_Analysis::BilgePumpVerification {
    subject sys;
    doc /* POSITIVE TEST — Nominal values
         ─────────────────────────────────────────────────────
         Q_net = (0.025 + 0.025) × 0.82 × 0.95 = 0.0389 m³/s

         BPS-REQ-001  waterLevel 0.15 ≤ 0.30 m          SATISFIED
         BPS-REQ-002  pumpB.isRedundant == true          SATISFIED
         BPS-REQ-003  alarmDelay 0.50 ≤ 2.00 s          SATISFIED
         BPS-REQ-004  Q_net 0.0389 ≥ 0.030 m³/s         SATISFIED
         ─────────────────────────────────────────────────────
         Overall: ALL SATISFIED
         For colored HTML output open Results.ipynb (Python kernel) */
    in attribute pumpAFlowRate = 0.025;
    in attribute pumpBFlowRate = 0.025;
    in attribute pumpEfficiency = 0.82;
    in attribute pumpARunHours = 120.0;
    in attribute pumpBRunHours = 85.0;
    in attribute pipeLossFactor = 0.05;
    in attribute designInflow = 0.030;
}


AnalysisCaseUsage positiveTest (68ea5ade-58b5-4d42-89f4-a050a268f3e1)


In [7]:
// ============================================================
// Cell 6 — NEGATIVE TEST: Pump A failure
//
// pumpA.flowRate overridden to 0.0 (pump A offline / isolated).
// Q_net = (0.0 + 0.025) × 0.82 × 0.95 = 0.0195 m³/s < 0.030
// ============================================================

analysis negativeTest : BilgePump_Analysis::BilgePumpVerification {
    subject sys;
    doc /* NEGATIVE TEST — Pump A offline (flowRate = 0.0)
         ─────────────────────────────────────────────────────
         Q_net = (0.0 + 0.025) × 0.82 × 0.95 = 0.0195 m³/s

         BPS-REQ-001  waterLevel 0.15 ≤ 0.30 m          SATISFIED
         BPS-REQ-002  pumpB.isRedundant == true          SATISFIED
         BPS-REQ-003  alarmDelay 0.50 ≤ 2.00 s          SATISFIED
         BPS-REQ-004  Q_net 0.0195 ≥ 0.030 m³/s         VIOLATED ✗
         ─────────────────────────────────────────────────────
         Overall: FAILURE DETECTED — BPS-REQ-004 violated
         Implication (SOLAS II-1 Reg. 35): pump B alone is
         insufficient; pump B must be upsized or a 3rd pump added. */
    in attribute pumpAFlowRate = 0.0;
    in attribute pumpBFlowRate = 0.025;
    in attribute pumpEfficiency = 0.82;
    in attribute pumpARunHours = 120.0;
    in attribute pumpBRunHours = 85.0;
    in attribute pipeLossFactor = 0.05;
    in attribute designInflow = 0.030;
}


AnalysisCaseUsage negativeTest (53d1c965-4f33-4f82-9986-822861c82984)


## Advanced SysML v2 Extensions

Three progressive extensions demonstrating more complex SysML v2 features on the same bilge pump model.

| Cell | Feature | SysML v2 Element |
|---|---|---|
| Phase A | Behavioral state machine for `PumpController` | `state def` + transitions |
| Phase B | Parametric sweep — pipe loss sensitivity | `calc def` with 5 instantiations |
| Phase C | Failure mode analysis — 3 fault scenarios | `analysis` specialization |

Expected verdicts for each scenario are embedded as `doc` blocks in the model elements.
Open **`Results.ipynb`** (Python kernel) for colored HTML pass/fail tables across all scenarios.


In [8]:
// ============================================================
// Phase A — Behavioral: PumpController State Machine
//
// Models the controller's discrete operating states and the
// guard conditions that drive transitions.
//
// States:
//   idle       — waiting, no alarm
//   sensing    — actively reading sensor input
//   activating — command sent to pump(s), awaiting flow confirmation
//   running    — pumps active, bilge discharging
//   alarm      — water level critical or pump fault detected
//
// Attributes declared on the state def are bound when the
// state machine is used on a PumpController part.
// ============================================================

package BilgePump_Behavioral {
    private import BilgePump_Library::*;
    private import ScalarValues::*;

    state def PumpControllerSM {
        // Observed values — bound at instantiation to PumpController attributes
        attribute triggerLevel_m  : Real;  // level that activates pumping (m)
        attribute responseTime_s  : Real;  // max allowed pump activation delay (s)
        attribute waterLevel      : Real;  // live sensor reading (m)
        attribute overrideActive  : Boolean; // operator manual override flag

        entry; then idle;

        // ── Idle ──────────────────────────────────────────────────────────
        state idle {
            entry action { }
        }

        // Unconditional: always begin sensing from idle
        transition idle_to_sensing
            first idle
            then sensing;

        // ── Sensing ───────────────────────────────────────────────────────
        state sensing {
            entry action { }
        }

        transition sensing_to_activating
            first sensing
            if triggerLevel_m > 0.0
            then activating;

        transition sensing_to_idle
            first sensing
            if triggerLevel_m == 0.0
            then idle;

        // ── Activating ────────────────────────────────────────────────────
        state activating {
            entry action { }
        }

        transition activating_to_running
            first activating
            if responseTime_s <= 1.0
            then running;

        // ── Running ───────────────────────────────────────────────────────
        state running {
            entry action { }
        }

        transition running_to_alarm
            first running
            if waterLevel > 0.25
            then alarm;

        transition running_to_idle
            first running
            if waterLevel <= 0.10
            then idle;

        // ── Alarm ─────────────────────────────────────────────────────────
        state alarm {
            entry action { }
        }

        transition alarm_to_idle
            first alarm
            if overrideActive == true
            then idle;
    }
}


Package BilgePump_Behavioral (fa10f157-24be-4ab3-acb4-e2a0f2ab74cf)


In [9]:
// ============================================================
// Phase B — Parametric sweep: pipe loss sensitivity
//
// calc def FlowSweep evaluates Q_net at 5 discrete pipe-loss
// values (0%, 5%, 10%, 15%, 20%) against the 0.030 m³/s inflow.
//
// This models the question: "at what pipe degradation does the
// combined pump capacity fall below the design inflow?"
//
// Combined flow: Q_A + Q_B = 0.025 + 0.025 = 0.050 m³/s
// η = 0.82
// Breakeven: λ = 1 − (designInflow / (Q_gross × η))
//           = 1 − (0.030 / 0.041) ≈ 0.268  → fails above ~27% loss
// ============================================================

package BilgePump_Parametric {
    private import BilgePump_Analysis::*;
    private import ScalarValues::*;

    calc def FlowSweep {
        in attribute flowRateA   : Real;
        in attribute flowRateB   : Real;
        in attribute efficiency  : Real;
        in attribute designInflow : Real;
        in attribute pipeLoss    : Real;   // swept variable

        // Returns Q_net for this pipe loss value
        return attribute q_net : Real = (flowRateA + flowRateB) * efficiency * (1.0 - pipeLoss);
    }

    // --- Sweep instantiations at 5 operating points ---

    calc sweepAt0pct : FlowSweep {
        in attribute flowRateA   = 0.025;
        in attribute flowRateB   = 0.025;
        in attribute efficiency  = 0.82;
        in attribute designInflow = 0.030;
        in attribute pipeLoss    = 0.00;
    }

    calc sweepAt5pct : FlowSweep {
        in attribute flowRateA   = 0.025;
        in attribute flowRateB   = 0.025;
        in attribute efficiency  = 0.82;
        in attribute designInflow = 0.030;
        in attribute pipeLoss    = 0.05;
    }

    calc sweepAt10pct : FlowSweep {
        in attribute flowRateA   = 0.025;
        in attribute flowRateB   = 0.025;
        in attribute efficiency  = 0.82;
        in attribute designInflow = 0.030;
        in attribute pipeLoss    = 0.10;
    }

    calc sweepAt15pct : FlowSweep {
        in attribute flowRateA   = 0.025;
        in attribute flowRateB   = 0.025;
        in attribute efficiency  = 0.82;
        in attribute designInflow = 0.030;
        in attribute pipeLoss    = 0.15;
    }

    calc sweepAt20pct : FlowSweep {
        in attribute flowRateA   = 0.025;
        in attribute flowRateB   = 0.025;
        in attribute efficiency  = 0.82;
        in attribute designInflow = 0.030;
        in attribute pipeLoss    = 0.20;
    }
}


Package BilgePump_Parametric (bdd172e4-59ec-411a-8f7a-3df16545334e)


In [10]:
// ============================================================
// Phase C — Failure Mode Analysis (FMEA)
//
// Three fault specializations of BilgePumpVerification, each
// overriding one or more parameters to model a discrete fault.
//
// Fault   Description                  REQ-004 result
// ──────  ───────────────────────────  ─────────────────────────
// A       Pump A offline               VIOLATED (0.0195 < 0.030)
// B       Both pumps offline           VIOLATED (0.0000 < 0.030)
// C       Sensor stuck-at-zero         SATISFIED (pumps run OK)
//
// SOLAS II-1 Reg. 35 finding from Fault A: pump B alone cannot
// meet design inflow — pump B must be upsized or a 3rd pump added.
// ============================================================

package BilgePump_FMEA {
    private import BilgePump_Analysis::*;
    private import ScalarValues::*;

    // ── Fault A: Pump A offline ───────────────────────────────────────
    analysis faultA_PumpAOffline : BilgePump_Analysis::BilgePumpVerification {
        subject sys;
        doc /* FAULT A — Pump A offline (pumpAFlowRate = 0.0)
             Q_net = (0.0 + 0.025) × 0.82 × 0.95 = 0.0195 m³/s
             REQ-001 SATISFIED  REQ-002 SATISFIED
             REQ-003 SATISFIED  REQ-004 VIOLATED ✗ */
        in attribute pumpAFlowRate   = 0.0;
        in attribute pumpBFlowRate   = 0.025;
        in attribute pumpEfficiency  = 0.82;
        in attribute pumpARunHours   = 120.0;
        in attribute pumpBRunHours   = 85.0;
        in attribute pipeLossFactor  = 0.05;
        in attribute designInflow    = 0.030;
    }

    // ── Fault B: Both pumps offline (SOLAS worst case) ────────────────
    analysis faultB_BothPumpsOffline : BilgePump_Analysis::BilgePumpVerification {
        subject sys;
        doc /* FAULT B — Both pumps offline (pumpA = 0.0, pumpB = 0.0)
             Q_net = 0.0 m³/s
             REQ-001 SATISFIED  REQ-002 SATISFIED
             REQ-003 SATISFIED  REQ-004 VIOLATED ✗ */
        in attribute pumpAFlowRate   = 0.0;
        in attribute pumpBFlowRate   = 0.0;
        in attribute pumpEfficiency  = 0.82;
        in attribute pumpARunHours   = 120.0;
        in attribute pumpBRunHours   = 85.0;
        in attribute pipeLossFactor  = 0.05;
        in attribute designInflow    = 0.030;
    }

    // ── Fault C: Sensor stuck-at-zero ─────────────────────────────────
    // Pumps run normally (physical state OK); sensor sends false-zero
    // to controller. Tests whether alarm fires independently of sensor.
    analysis faultC_SensorFault : BilgePump_Analysis::BilgePumpVerification {
        subject sys;
        doc /* FAULT C — Sensor stuck-at-zero (waterLevel reads 0 m)
             Q_net = (0.025 + 0.025) × 0.82 × 0.95 = 0.0389 m³/s
             REQ-001 SATISFIED (sensor reads 0 ≤ 0.30 — false pass)
             REQ-002 SATISFIED  REQ-003 SATISFIED
             REQ-004 SATISFIED (pumps running normally)
             Note: a diagnostic requirement should assert
             sensor.waterLevel != 0 while alarm.isActive == true. */
        in attribute pumpAFlowRate   = 0.025;
        in attribute pumpBFlowRate   = 0.025;
        in attribute pumpEfficiency  = 0.82;
        in attribute pumpARunHours   = 120.0;
        in attribute pumpBRunHours   = 85.0;
        in attribute pipeLossFactor  = 0.05;
        in attribute designInflow    = 0.030;
    }
}


Package BilgePump_FMEA (f19d51de-7614-462c-a0fe-160d52b00ad4)


In [ ]:
// =============================================================================
// Cell 7a — Publish Library to SST API
// IMPORTANT: Each %publish call must be in its own cell.
// Bare "%publish" with no element name does nothing.
// =============================================================================
%publish BilgePump_Library

In [ ]:
// Cell 7b — Publish Architecture to SST API
%publish BilgePump_Architecture

In [ ]:
// Cell 7c — Publish Requirements to SST API
%publish BilgePump_Requirements

In [ ]:
// Cell 7d — Publish Analysis to SST API
%publish BilgePump_Analysis